In [4]:
# !pip install -q transformers datasets peft accelerate evaluate sacrebleu sentencepiece bitsandbytes

In [3]:
import torch
import glob
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
import evaluate

In [5]:
# Load and Combine XLSX Files

xlsx_files = glob.glob("*.xlsx")
processed_dfs = []

for file in xlsx_files:
    df = pd.read_excel(file)
    df.columns = df.columns.str.strip()

    # Identify dialect and standard columns dynamically per file
    dialect_cols = [c for c in df.columns if 'dialect' in c.lower()]
    standard_cols = [c for c in df.columns if 'standard' in c.lower()]

    if dialect_cols and standard_cols:
      d_col = dialect_cols[0]
      s_col = standard_cols[0]

      temp_df = pd.DataFrame()
      temp_df['source'] = df[d_col].astype(str).str.strip()
      temp_df['target'] = df[s_col].astype(str).str.strip()

      temp_df['dialect_type'] = d_col.replace("Dialect", '').strip()

      temp_df = temp_df.dropna()
      temp_df = temp_df[(temp_df['source'] != '') & (temp_df['target'] != '') & (temp_df['source'] != 'nan')]

      processed_dfs.append(temp_df)

full_df = pd.concat(processed_dfs, ignore_index=True)
print(f"Dataset successfully loaded with {len(full_df)} total rows.")
print("Distribution per dialect:\n", full_df['dialect_type'].value_counts())




Dataset successfully loaded with 17329 total rows.
Distribution per dialect:
 dialect_type
Sylheti         9320
Chittagonian    8009
Name: count, dtype: int64


In [6]:
train_df, temp_df = train_test_split(
    full_df,
    test_size=0.20,
    random_state=42,
    stratify=full_df['dialect_type']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df['dialect_type']
)

dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df[['source', 'target', 'dialect_type']], preserve_index=False),
    'validation': Dataset.from_pandas(val_df[['source', 'target', 'dialect_type']], preserve_index=False),
    'test': Dataset.from_pandas(test_df[['source', 'target', 'dialect_type']], preserve_index=False)
})

print("\nDataset splits created successfully:")
print(dataset)


Dataset splits created successfully:
DatasetDict({
    train: Dataset({
        features: ['source', 'target', 'dialect_type'],
        num_rows: 13863
    })
    validation: Dataset({
        features: ['source', 'target', 'dialect_type'],
        num_rows: 1733
    })
    test: Dataset({
        features: ['source', 'target', 'dialect_type'],
        num_rows: 1733
    })
})


In [7]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG = "ben_Beng"  # Bengali script source
TGT_LANG = "ben_Beng"  # Standard Bengali target

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    src_lang=SRC_LANG,
    tgt_lang=TGT_LANG
)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/3.55k [00:00<?, ?B/s]

In [8]:
# 8-bit model
if torch.cuda.is_available():
    quantization_config = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map={"": 0}   # single GPU
    )
else:
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)



model = prepare_model_for_kbit_training(model)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.46GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [9]:
# LoRA
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"]
)

model = get_peft_model(model, peft_config=peft_config)
model.print_trainable_parameters()
model.config.use_cache = False

trainable params: 4,718,592 || all params: 619,792,384 || trainable%: 0.7613


In [10]:
# tokenization

max_input_length = 128
max_target_length = 128

def preprocess_function(examples):
    inputs = examples['source']
    targets = examples['target']

    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    labels = tokenizer(text_target=targets, max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/13863 [00:00<?, ? examples/s]

Map:   0%|          | 0/1733 [00:00<?, ? examples/s]

Map:   0%|          | 0/1733 [00:00<?, ? examples/s]

In [11]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Replace -100 in labels (used for padding) before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [[l.strip()] for l in decoded_labels]  # sacrebleu wants list-of-lists

    bleu = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    chrf = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "bleu": bleu["score"],
        "chrf": chrf["score"],
    }

In [12]:
import warnings
import logging
import torch
import logging

import os
import ctypes
import warnings
import logging

#  Disable bitsandbytes internal C++ logging
os.environ["BITSANDBYTES_NOWELCOME"] = "1"

try:
    import bitsandbytes as bnb
    # Mute the bitsandbytes autograd warnings module directly
    bnb.autograd._functions.logger.setLevel(logging.ERROR)
except Exception:
    pass

#  Suppress Python-level warning triggers
warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("transformers").setLevel(logging.ERROR)



# Save trained adapter weights locally
model.save_pretrained("./final_nllb_dialect_lora")
tokenizer.save_pretrained("./final_nllb_dialect_lora")

# Silence bitsandbytes warnings
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)

# Silence transformers warnings completely
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()


# 1. Suppress bitsandbytes output spam inside the active runtime scope
warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("transformers").setLevel(logging.ERROR)




# training arguments
# training_args = Seq2SeqTrainingArguments(
#     output_dir="./nllb_dialect_adapter",
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     learning_rate=2e-4,
#     per_device_train_batch_size=16,   # reduce for debugging
#     per_device_eval_batch_size=16,
#     weight_decay=0.01,
#     save_total_limit=2,
#     num_train_epochs=5,              # reduce for debugging
#     predict_with_generate=True,     # disable for debugging
#     fp16=torch.cuda.is_available(),
#     logging_steps=1,                 # log every step
#     report_to="none"
# )




# data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# bleu_metric = evaluate.load("sacrebleu")
# chrf_metric = evaluate.load("chrf")

# trainer = Seq2SeqTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_datasets["train"],
#     eval_dataset=tokenized_datasets["validation"],
#     processing_class=tokenizer,
#     data_collator=data_collator,
#     compute_metrics=compute_metrics,   # <-- new
# )
# # Execute Fine-Tuning
# trainer.train()

# # Save trained adapter weights locally
# model.save_pretrained("./final_nllb_dialect_lora")
# tokenizer.save_pretrained("./final_nllb_dialect_lora")






training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_dialect_adapter",
    eval_strategy="epoch",            # Evaluate once at epoch end
    save_strategy="epoch",            # Save once at epoch end
    learning_rate=5e-4,
    per_device_train_batch_size=16,   # Increased from 1 to maximize T4 GPU VRAM utilization
    per_device_eval_batch_size=16,    # Process validation set in larger parallel batches
    gradient_accumulation_steps=1,    # Direct updates
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=10,
    predict_with_generate=False,      # FAST EVALUATION: Calculates loss without slow autoregressive token generation
    fp16=True,                        # Mixed precision on T4 CUDA cores
    logging_steps=20,                 # Log metrics every 20 steps instead of every 1
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    # Note: compute_metrics is omitted here because predict_with_generate=False
    # computes fast CrossEntropy loss during validation. You will run BLEU/chrF on test set after training.
)

# Clear unused VRAM memory
torch.cuda.empty_cache()

# Execute Fine-Tuning
trainer.train()

# Save trained adapter weights locally
model.save_pretrained("./final_nllb_dialect_lora")
tokenizer.save_pretrained("./final_nllb_dialect_lora")

{'loss': '2.228', 'grad_norm': '0.5591', 'learning_rate': '0.0004989', 'epoch': '0.02307'}
{'loss': '1.471', 'grad_norm': '0.4217', 'learning_rate': '0.0004978', 'epoch': '0.04614'}
{'loss': '1.345', 'grad_norm': '0.5228', 'learning_rate': '0.0004966', 'epoch': '0.0692'}
{'loss': '1.277', 'grad_norm': '0.5309', 'learning_rate': '0.0004954', 'epoch': '0.09227'}
{'loss': '1.219', 'grad_norm': '0.5196', 'learning_rate': '0.0004943', 'epoch': '0.1153'}
{'loss': '1.167', 'grad_norm': '0.5112', 'learning_rate': '0.0004931', 'epoch': '0.1384'}
{'loss': '1.183', 'grad_norm': '0.5567', 'learning_rate': '0.000492', 'epoch': '0.1615'}
{'loss': '1.153', 'grad_norm': '0.5904', 'learning_rate': '0.0004908', 'epoch': '0.1845'}
{'loss': '1.088', 'grad_norm': '0.4457', 'learning_rate': '0.0004897', 'epoch': '0.2076'}
{'loss': '1.071', 'grad_norm': '0.6169', 'learning_rate': '0.0004885', 'epoch': '0.2307'}
{'loss': '1.082', 'grad_norm': '0.7621', 'learning_rate': '0.0004874', 'epoch': '0.2537'}
{'loss':

('./final_nllb_dialect_lora/tokenizer_config.json',
 './final_nllb_dialect_lora/tokenizer.json')

In [13]:
import pandas as pd
import torch
import evaluate

bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

test_sources = dataset["test"]["source"]
test_targets = dataset["test"]["target"]
test_dialects = dataset["test"]["dialect_type"]

# Fetch target language BOS token ID safely for NLLB
target_lang_id = tokenizer.convert_tokens_to_ids(TGT_LANG)

predictions = []
batch_size = 8

model.eval()

# Generate translations for test set
for i in range(0, len(test_sources), batch_size):
    batch_inputs = test_sources[i:i + batch_size]
    inputs = tokenizer(batch_inputs, padding=True, truncation=True, max_length=128, return_tensors="pt").to("cuda")

    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=target_lang_id,
            max_length=128
        )

    decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    predictions.extend(decoded_preds)

# Create DataFrame for results analysis
results_df = pd.DataFrame({
    'dialect': test_dialects,
    'source': test_sources,
    'target': test_targets,
    'prediction': predictions
})

# Overall Scores
references = [[tgt] for tgt in test_targets]
overall_bleu = bleu_metric.compute(predictions=predictions, references=references)
overall_chrf = chrf_metric.compute(predictions=predictions, references=references)

print(f"\n==========================================")
print(f"--- OVERALL TEST RESULTS ---")
print(f"BLEU: {overall_bleu['score']:.2f}")
print(f"chrF: {overall_chrf['score']:.2f}")
print(f"==========================================")

# Breakdown by Dialect
print("\n--- PER-DIALECT BREAKDOWN ---")
for dialect_name in results_df['dialect'].unique():
    subset = results_df[results_df['dialect'] == dialect_name]
    sub_preds = subset['prediction'].tolist()
    sub_refs = [[tgt] for tgt in subset['target'].tolist()]

    sub_bleu = bleu_metric.compute(predictions=sub_preds, references=sub_refs)
    sub_chrf = chrf_metric.compute(predictions=sub_preds, references=sub_refs)

    print(f"\nDialect: {dialect_name} (Samples: {len(subset)})")
    print(f"  BLEU: {sub_bleu['score']:.2f}")
    print(f"  chrF: {sub_chrf['score']:.2f}")

# Sample Predictions
print("\n--- SAMPLE PREDICTIONS ---")
print(results_df[['dialect', 'source', 'prediction', 'target']].head(6).to_string())


--- OVERALL TEST RESULTS ---
BLEU: 50.27
chrF: 76.41

--- PER-DIALECT BREAKDOWN ---

Dialect: Sylheti (Samples: 932)
  BLEU: 44.93
  chrF: 72.58

Dialect: Chittagonian (Samples: 801)
  BLEU: 55.84
  chrF: 80.63

--- SAMPLE PREDICTIONS ---
        dialect                                                                   source                                                                 prediction                                                                   target
0       Sylheti  অংকের ভাষায়— একজন ম্যাসাজম্যান গা টিপ্পা অন্য এখজনরে শারীরিক আরাম দিরো।  অংকের ভাষায় একজন ম্যাসাজম্যান গা টিপ্পা অন্য একজনকে শারীরিক আরাম দিচ্ছে।  অংকের ভাষায়— একজন ম্যাসাজম্যান গা টিপে অন্য একজনকে শারীরিক আরাম দিচ্ছে।
1  Chittagonian                           পৃথিবীর প্রতি আঁর  মমতা আছে বুলিই এই ধর্মপালন।                              পৃথিবীর প্রতি আমার মমতা আছে বলেই এই ধর্মপালন।                            পৃথিবীর প্রতি আমার মমতা আছে বলেই এই ধর্মপালন।
2  Chittagonian                                আঁ

In [14]:
full_df

,source,target,dialect_type
0,বিয়ার ফরেও কিতা আমার এখলা আটা লাগব নি?,নাকি বিয়ার পরেও আমাকে একা হাঁটতে হবে?,Sylheti
1,"প্রিয় ফাঠক, আমি যে মুবাইলো খতা খইরাম ওটা আমার ...","প্রিয় পাঠক, যে মোবাইলে আমি কথা বলছি সেটা আমার না।",Sylheti
2,"অখন থাকি তুই আমারে ফরামর্শ দিবে,বুচ্ছতনি।","এখন থেকে তুই আমাকে পরামর্শ দিবি, বুঝলি।",Sylheti
3,"তাইন,কোয়াই?",উনি কোথায়?,Sylheti
4,হে মুন্সিগঞ্জও তার মার বাড়িত গেছে।,সে মুন্সিগঞ্জে তার মায়ের বাড়িতে গেছে।,Sylheti
...,...,...,...
17324,কিন্তু কামর চাবে ইতারে তাড়াতাড়ি বড় হন পইজ্জি ।,কিন্তু কাজের চাপে তাকে দ্রুত বড় হতে হয়েছে।,Chittagonian
17325,রতন এই স্বল্পশিক্ষিত যুবকইবারে তার জীবনর একমাত...,রতন এই স্বল্পশিক্ষিত যুবকটিকে তার জীবনের একমাত...,Chittagonian
17326,ইতা যেন তার উগগো খেলার ওঙ্গিয়া হোই উইটঠি।,সে যেন তার এক খেলার সাথী হয়ে উঠেছিল।,Chittagonian
17327,গেরাম ছাড়ি যাইবার খবর হুনি রতন ভিতরে ভিতরে কষ্...,গ্রাম ছেড়ে যাওয়ার খবর শুনে রতন ভেতরে ভেতরে কষ...,Chittagonian


Generated Code

In [15]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import torch

# multilingual model perfect for Bengali
embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Prepare the knowledge base from your training data
# We use the dialect source as the key, and standard target as the value
kb_df = train_df[['source', 'target', 'dialect_type']].copy().reset_index(drop=True)

# Create embeddings for all dialect sentences in the KB
print("Embedding knowledge base... (this may take a minute)")
kb_embeddings = embedder.encode(kb_df['source'].tolist(), show_progress_bar=True)
print("Knowledge base embedded!")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding knowledge base... (this may take a minute)


Batches:   0%|          | 0/434 [00:00<?, ?it/s]

Knowledge base embedded!


In [16]:
# Retriever
def retrieve_similar(query, dialect_type,  top_k=3):
  """Retrieves top_k most similar examples from the KB for a given dialect."""
  # Filter KB to only match the dialect
  dialect_mask = kb_df['dialect_type'] == dialect_type
  filtered_kb = kb_df[dialect_mask]
  filtered_embeddings = kb_embeddings[dialect_mask]

  if len(filtered_kb) == 0:
    return pd.DataFrame()


  query_emb = embedder.encode([query])
  similarities = cosine_similarity(query_emb, filtered_embeddings)[0]
  top_indices = np.argsort(similarities)[-top_k:][::-1]

  return filtered_kb.iloc[top_indices]


def build_rag_prompt(query, dialect_type, retrieved_examples):
  """Formats the retrieved examples into a prompt optimized for NLLB."""

  prompt = ""

  if not retrieved_examples.empty:
    for _, row in retrieved_examples.iterrows():
      prompt += f"{row['source']} -> {row['target']}\n"

    prompt += f"{query} ->"
    return prompt



In [17]:
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

test_sources = dataset["test"]["source"]
test_targets = dataset["test"]["target"]
test_dialects = dataset["test"]["dialect_type"]

target_lang_id = tokenizer.convert_tokens_to_ids(TGT_LANG)

rag_predictions = []
batch_size = 4  # reduced for rag

print("Generating RAG translations...")

for i in range(0, len(test_sources), batch_size):
  batch_sources = test_sources[i:i + batch_size]
  batch_dialects = test_dialects[i:i + batch_size]

  rag_prompts = []
  for src, dialect in zip(batch_sources, batch_dialects):
    retrieved = retrieve_similar(src, dialect, top_k=3)
    prompt = build_rag_prompt(src, dialect, retrieved)
    rag_prompts.append(prompt)

  # increasing max length to 256 to accomodate the retrieved examples
  inputs = tokenizer(rag_prompts, padding=True, truncation=True, max_length=256, return_tensors="pt").to("cuda")


  with torch.no_grad():
    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=target_lang_id,
        max_length=128
    )

  decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
  rag_predictions.extend(decoded_preds)


# Create DataFrame for results
rag_results_df = pd.DataFrame({
    'dialect': test_dialects,
    'source': test_sources,
    'target': test_targets,
    'rag_prediction': rag_predictions
})


# overall scores
references = [[tgt] for tgt in test_targets]
overall_bleu = bleu_metric.compute(predictions=rag_predictions, references=references)
overall_chrf = chrf_metric.compute(predictions=rag_predictions, references=references)

print(f"\n==========================================")
print(f"--- OVERALL RAG TEST RESULTS ---")
print(f"BLEU: {overall_bleu['score']:.2f}")
print(f"chrF: {overall_chrf['score']:.2f}")
print(f"==========================================")

# Sample Predictions
print("\n--- SAMPLE RAG PREDICTIONS ---")
# pd.set_option('display.max_colwidth', None) # Uncomment if you want to see full text
print(rag_results_df[['dialect', 'source', 'rag_prediction', 'target']].head(6).to_string())


Generating RAG translations...

--- OVERALL RAG TEST RESULTS ---
BLEU: 2.74
chrF: 26.08

--- SAMPLE RAG PREDICTIONS ---
        dialect                                                                   source                                                                                                                                                                                                                                                                                                                                                                                                             rag_prediction                                                                   target
0       Sylheti  অংকের ভাষায়— একজন ম্যাসাজম্যান গা টিপ্পা অন্য এখজনরে শারীরিক আরাম দিরো।  হাত দেখার বিষয়টা এই মহিলা সিরিয়াসলি নিয়েছেন তা বোঝা যাচ্ছে। -> হাত দেখার বিষয়টা এই মহিলা সিরিয়াসলি নিয়েছেন তা বোঝা যাচ্ছে। যে কোন তুচ্ছ ব্যাপারে দুঃশ্চিন্তা করে ফাতেমা খালা অসুখ বধিয়ে ফেললেন। -> যে কোন তুচ্ছ ব্